<a href="https://colab.research.google.com/github/AaryeshShukla/Scheduled-Scraping-with-GitHub-Actions/blob/main/Minor_project_github_pus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [118]:
!pip install flask requests python-dotenv pyngrok


In [119]:
!pkill ngrok


In [120]:
from pyngrok import ngrok

ngrok.set_auth_token("33eQN8nQWJuzqbha7iyEOgDtF2e_73YLEQNp6VrHL67VfR3ug")

public_url = ngrok.connect(5000)
print(public_url)


NgrokTunnel: "https://prefixally-piazzaed-alverta.ngrok-free.dev" -> "http://localhost:5000"


In [121]:
import os
from getpass import getpass

# Set your AI Pipe token securely
os.environ["AI_PIPE_TOKEN"] = getpass("Enter your AI Pipe token: ")


Enter your AI Pipe token: ··········


In [122]:
%%writefile app.py
from flask import Flask, request
import requests
import json
import os
import re
import base64

app = Flask(__name__)

# Simple HTML page
HTML = """
<h1>RepoGen AI Pipe + GitHub</h1>
<p>Enter your prompt and get AI-generated code pushed to GitHub:</p>
<form method="post" action="/generate">
<p>GitHub Token: <input type="text" name="github_token" required></p>
<textarea name="prompt" rows="4" cols="50" placeholder="Enter prompt here..." required></textarea><br><br>
<button>Generate & Push to GitHub</button>
</form>
"""

@app.route("/")
def home():
    return HTML

@app.route("/generate", methods=["POST"])
def generate():
    prompt = request.form.get("prompt", "")
    github_token = request.form.get("github_token", "")
    if not prompt or not github_token:
        return "Please enter prompt and GitHub token!"

    # AI Pipe token from environment
    AI_PIPE_TOKEN = os.environ.get("AI_PIPE_TOKEN")
    if not AI_PIPE_TOKEN:
        return "AI Pipe token not set in environment variables!"

    try:
        # --- Generate code with AI Pipe ---
        headers = {
            "Authorization": f"Bearer {AI_PIPE_TOKEN}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": "openai/gpt-4.1-nano",
            "messages": [
                {
                    "role": "user",
                    "content": f"Generate code for this prompt:\n{prompt}\nReturn only code in JSON format: {{'files': {{'main.py': '...'}}}}"
                }
            ],
            "temperature": 0.2
        }

        AI_ENDPOINT = "https://aipipe.org/openrouter/v1/chat/completions"
        response = requests.post(AI_ENDPOINT, headers=headers, json=payload)

        if response.status_code != 200:
            return f"AI API Error: {response.status_code} {response.text}"

        data = response.json()
        ai_content = data["choices"][0]["message"]["content"]

        # Extract JSON safely
        match = re.search(r'\{.*\}', ai_content, re.DOTALL)
        if not match:
            return f"Could not find JSON in AI response:<br>{ai_content}"

        files = json.loads(match.group(0))["files"]

        # --- Create GitHub repo ---
        repo_name = f"repo_{str(abs(hash(prompt)))[:8]}"
        gh_headers = {
            "Authorization": f"token {github_token}",
            "Accept": "application/vnd.github.v3+json"
        }

        repo_data = {"name": repo_name, "private": False}
        repo_resp = requests.post("https://api.github.com/user/repos", headers=gh_headers, json=repo_data)

        if repo_resp.status_code not in [200, 201]:
            return f"GitHub repo creation failed: {repo_resp.status_code} {repo_resp.text}"

        # --- Upload files ---
        for filename, content in files.items():
            file_data = {
                "message": f"Add {filename}",
                "content": base64.b64encode(content.encode()).decode()
            }
            url = f"https://api.github.com/repos/{repo_resp.json()['owner']['login']}/{repo_name}/contents/{filename}"
            file_resp = requests.put(url, headers=gh_headers, json=file_data)
            if file_resp.status_code not in [200, 201]:
                return f"Failed to upload {filename}: {file_resp.status_code} {file_resp.text}"

        return f"<h2>Repo created successfully!</h2><p>GitHub repo: <a href='{repo_resp.json()['html_url']}' target='_blank'>{repo_name}</a></p>"

    except Exception as e:
        return f"Server error: {str(e)}"

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=True)


Overwriting app.py


In [123]:
!kill -9 $(lsof -t -i:5000)


In [124]:
!nohup python app.py &


nohup: appending output to 'nohup.out'


In [125]:
from pyngrok import ngrok

public_url = ngrok.connect(5000)
print("🌐 Public URL:", public_url)


🌐 Public URL: NgrokTunnel: "https://prefixally-piazzaed-alverta.ngrok-free.dev" -> "http://localhost:5000"
